## __Data Engineering__
- Sau khi cleaning pipeline hoàn tất và dữ liệu đã được chuẩn hóa về booking-level ở phần [data_cleanning](python/data_cleaning.ipynb), bước tiếp theo là tạo các analytical features phục vụ KPI và EDA.
- Feature Engineering được thực hiện trên `df_analytical`.

#### __1.1 Core Delivery Performance Features__
Ở phần này các feature chính được xây dựng gồm:
- `delivery_variance_hours`: chênh lệch giữa Actual ETA và Planned ETA.
- `delay_hours`: số giờ giao trễ, không nhận giá trị âm.
- `is_delayed`: booking có giao trễ hay không.
- `is_ontime`: booking giao đúng hoặc sớm hơn kế hoạch.
- `delivery_status`: trạng thái dễ đọc phục vụ phân tích và dashboard.

Trong đó, những booking không đủ điều kiện tính KPI sẽ giữ missing ở các feature này thay vì bị tự động coi là On Time.

In [45]:
import pandas as pd
import numpy as np
import re

In [2]:
df_analytical = pd.read_excel('../data/processed/logistics_tracking_cleaned.xlsx')

In [3]:
# Tạo mask cho booking đủ điều kiện
eligible_mask = (df_analytical["valid_kpi_timeline"])

print("Eligible bookings:", eligible_mask.sum())
print("Ineligible bookings:",(~eligible_mask).sum())

Eligible bookings: 3548
Ineligible bookings: 34


##### Tính Delivery Variance Hours
**Delivery Variance Hours = Actual ETA - Planned ETA**

In [4]:
df_analytical.dtypes

booking_id                            object
shipment_type                         object
booking_date                  datetime64[ns]
vehicle_registration                  object
vehicle_type                          object
origin_location                       object
destination_location                  object
transportation_distance_km           float64
planned_eta                   datetime64[ns]
actual_eta                    datetime64[ns]
trip_start_date               datetime64[ns]
customer_name                         object
supplier_name                         object
material_shipped                      object
valid_kpi_timeline                      bool
invalid_timeline                        bool
kpi_eligibility_reason                object
distance_available                      bool
vehicle_type_known                      bool
multiple_customers                      bool
multiple_materials                      bool
delivery_variance_hours              float64
dtype: obj

In [5]:
df_analytical["delivery_variance_hours"] = pd.Series(pd.NA, index=df_analytical.index, dtype="Float64")


df_analytical.loc[eligible_mask, "delivery_variance_hours"] = (
    df_analytical.loc[eligible_mask, "actual_eta"] - df_analytical.loc[eligible_mask, "planned_eta"]
).dt.total_seconds().div(3600)

df_analytical["delivery_variance_hours"].describe().round(2)

count     3548.0
mean      147.12
std       473.89
min       -99.97
25%       -38.93
50%        35.16
75%       151.34
max      4977.42
Name: delivery_variance_hours, dtype: Float64

In [6]:
# Kiểm tra booking giao sớm
df_analytical.loc[df_analytical["delivery_variance_hours"] < 0,
                  [
                      "booking_id",
                      "planned_eta",
                      "actual_eta",
                      "delivery_variance_hours"
                  ]].head(10)

,booking_id,planned_eta,actual_eta,delivery_variance_hours
0,AEIBK2027469,2020-08-30 16:03:46.000,2020-08-28 12:48:07.380,-51.260728
1,VCV00014153/082021,2020-08-31 19:21:48.570,2020-08-28 12:13:51.543,-79.132508
2,VCV00014063/082021,2020-08-31 18:22:17.833,2020-08-28 11:33:19.043,-78.816331
3,VCV00014741/082021,2020-09-01 04:32:20.523,2020-08-28 11:31:29.780,-89.014095
4,AEIBK2027446,2020-08-30 13:55:08.000,2020-08-29 07:09:41.173,-30.757452
5,AEIBK2027346,2020-08-29 19:45:33.000,2020-08-28 11:45:03.900,-32.008083
6,VCV00014935/082021,2020-09-01 15:14:21.960,2020-08-28 12:00:38.340,-99.228783
7,VCV00014113/082021,2020-08-31 19:05:12.770,2020-08-28 11:04:55.737,-80.004731
8,VCV00013971/082021,2020-08-31 16:57:32.070,2020-08-28 10:48:38.800,-78.148131
9,AEIBK2027316,2020-08-29 17:39:48.000,2020-08-28 11:05:48.657,-30.566484


##### Delay Hours:
**Delay Hours = max(Delivery Variance Hours,0)**

In [7]:
df_analytical["delay_hours"] = pd.Series(
    pd.NA,
    index=df_analytical.index,
    dtype="Float64"
)

df_analytical.loc[eligible_mask, "delay_hours"] = (
    df_analytical.loc[eligible_mask, "delivery_variance_hours"].clip(lower=0)
)

print("Negative delay hours:",(df_analytical["delay_hours"] < 0).sum())

assert (df_analytical["delay_hours"].dropna().ge(0).all())

Negative delay hours: 0


> Như vậy:  
>Variance = -20 → Delay = 0  
>Variance =   0 → Delay = 0    
>Variance =  10 → Delay = 10  
>Variance = 120 → Delay = 120 

In [8]:
# Nếu Actual ETA > Planned ETA --> Delayed
df_analytical["is_delayed"] = pd.Series(
    pd.NA,
    index=df_analytical.index,
    dtype="Int64"
)

df_analytical.loc[eligible_mask,"is_delayed"] = (
    df_analytical.loc[eligible_mask, "delivery_variance_hours"].gt(0).astype("Int64")
)

In [9]:
# Nếu Actual ETA < Planned ETA --> Ontime
df_analytical["is_ontime"] = pd.Series(
    pd.NA,
    index=df_analytical.index,
    dtype="Int64"
)

df_analytical.loc[eligible_mask,"is_ontime"] = (
    df_analytical.loc[eligible_mask, "delivery_variance_hours"].le(0).astype("Int64")
)

In [10]:
# Validate is_delayed và is_ontime
status_sum = (df_analytical.loc[eligible_mask,"is_delayed"] + 
              df_analytical.loc[eligible_mask,"is_ontime"])

status_sum.value_counts()
assert status_sum.eq(1).all()

##### Tạo delivery_status

In [11]:
df_analytical["delivery_status"] = pd.Series(
    pd.NA,
    index=df_analytical.index,
    dtype="string"
)

# Delayed
df_analytical.loc[eligible_mask &
                  df_analytical["is_delayed"].eq(1),"delivery_status"] = "Delayed"

# Ontime
df_analytical.loc[eligible_mask &
    df_analytical["is_ontime"].eq(1),"delivery_status"] = "On Time"

df_analytical[ "delivery_status"].value_counts(dropna=False)

delivery_status
Delayed    2184
On Time    1364
<NA>         34
Name: count, dtype: Int64

In [12]:
df_analytical[
    [
        "booking_id",
        "planned_eta",
        "actual_eta",
        "valid_kpi_timeline",
        "delivery_variance_hours",
        "delay_hours",
        "is_delayed",
        "is_ontime",
        "delivery_status"
    ]
].sample(10,random_state=42)

,booking_id,planned_eta,actual_eta,valid_kpi_timeline,delivery_variance_hours,delay_hours,is_delayed,is_ontime,delivery_status
1407,AEIBK2025084,2020-08-11 13:49:01.000,2020-08-09 14:34:13.080,True,-47.246644,0.0,0,1,On Time
411,AEIBK2026867,2020-08-25 18:49:11.000,2020-08-24 12:36:44.240,True,-30.207433,0.0,0,1,On Time
802,VCV00008782/082021,2020-08-23 13:55:25.847,2020-08-19 10:13:20.763,True,-99.701412,0.0,0,1,On Time
3032,AEIBK2012507,2020-01-17 00:23:37.000,2020-03-13 18:24:55.420,True,1362.021783,1362.021783,1,0,Delayed
1546,AEIBK2020731,2020-06-21 23:51:35.000,2020-08-06 03:23:19.933,True,1083.529148,1083.529148,1,0,Delayed
554,AEIBK2026484,2020-08-16 03:33:28.000,2020-08-21 13:18:46.600,True,129.755167,129.755167,1,0,Delayed
810,VCV00008126/082021,2020-08-22 13:12:53.733,2020-08-19 09:02:11.320,True,-76.178448,0.0,0,1,On Time
2807,AEIBK2019882,2020-06-09 18:07:21.000,2020-06-18 18:05:11.927,True,215.964146,215.964146,1,0,Delayed
1068,AEIBK2025901,2020-08-18 01:13:27.000,2020-08-17 08:21:41.763,True,-16.862566,0.0,0,1,On Time
270,AEIBK2026929,2020-08-22 22:22:12.000,2020-08-25 18:06:12.733,True,67.733537,67.733537,1,0,Delayed


__Kết luận:__  
- Các core delivery performance features đã được tạo trên booking-level analytical dataset.

- `delivery_variance_hours` biểu diễn chênh lệch có dấu giữa Actual ETA và Planned ETA. Giá trị âm thể hiện booking đến sớm, giá trị bằng 0 thể hiện đến đúng ETA và giá trị dương thể hiện giao trễ.

- `delay_hours` chỉ biểu diễn phần giao trễ và không nhận giá trị âm. Booking đến sớm hoặc đúng ETA có Delay Hours bằng 0.

- `is_delayed`, `is_ontime` và `delivery_status` chỉ được tạo cho các booking có timeline đủ điều kiện tính KPI. Booking thiếu hoặc có timeline không hợp lệ không bị mặc định coi là On Time.

#### __1.2 Time Features__
Ở phần này ta sẽ tạo các biến thời gian để hỗ trợ phân tích xu hướng và seasonality.

##### Tạo Hours

In [13]:
# Tạo booking Hour
df_analytical["booking_hour"] = (df_analytical["booking_date"].dt.hour.astype("Int64"))

df_analytical["booking_hour"].value_counts().sort_index()

booking_hour
0      19
1      19
2       7
3       7
4       1
5       1
6       5
7      20
8      55
9     154
10    295
11    321
12    281
13    249
14    419
15    444
16    334
17    274
18    178
19    168
20    128
21     91
22     53
23     59
Name: count, dtype: Int64

##### Tạo Year

In [14]:
# Tạo booking year
df_analytical["booking_year"] = (
    df_analytical["booking_date"].dt.year.astype("Int64")
)

df_analytical["booking_year"].value_counts().sort_index()

booking_year
2019     184
2020    3398
Name: count, dtype: Int64

##### Tạo month

In [15]:
# Số tháng
df_analytical["booking_month_num"] = (
    df_analytical["booking_date"].dt.month.astype("Int64")
)

# Month start
df_analytical["booking_month_start"] = (
    df_analytical["booking_date"].dt.to_period("M").dt.to_timestamp()
)

# Month label
df_analytical["booking_month"] = (
    df_analytical["booking_date"].dt.strftime("%Y-%m").astype("string")
)


##### Tạo quarter

In [16]:
df_analytical["booking_quarter"] = (
    df_analytical["booking_date"].dt.to_period("Q").astype("string")
)

##### Tạo weekday

In [17]:
# Weekday number
df_analytical["booking_weekday_num"] = (
    df_analytical["booking_date"].dt.dayofweek.add(1).astype("Int64")
)

# Weekday name
df_analytical["booking_weekday"] = (
    df_analytical["booking_date"].dt.day_name().astype("string")
)

# Weekend flag
df_analytical["is_weekend_booking"] = (
    df_analytical["booking_weekday_num"].isin([6, 7]).astype("Int64")
)

In [18]:
# Tạo timeline features cho planned_eta
df_analytical["planned_delivery_date"] = (
    df_analytical["planned_eta"].dt.normalize()
)

df_analytical["planned_delivery_month"] = (
    df_analytical["planned_eta"].dt.to_period("M").dt.to_timestamp()
)

In [19]:
df_analytical[
    [
        "booking_id",
        "booking_date",
        "booking_hour",
        "booking_year",
        "booking_quarter",
        "booking_month",
        "booking_month_start",
        "booking_weekday",
        "booking_weekday_num",
        "is_weekend_booking",
        "planned_eta",
        "planned_delivery_month"
    ]
].head(10)

,booking_id,booking_date,booking_hour,booking_year,booking_quarter,booking_month,booking_month_start,booking_weekday,booking_weekday_num,is_weekend_booking,planned_eta,planned_delivery_month
0,AEIBK2027469,2020-08-26 12:03:46.000,12,2020,2020Q3,2020-08,2020-08-01,Wednesday,3,0,2020-08-30 16:03:46.000,2020-08-01
1,VCV00014153/082021,2020-08-27 15:21:48.570,15,2020,2020Q3,2020-08,2020-08-01,Thursday,4,0,2020-08-31 19:21:48.570,2020-08-01
2,VCV00014063/082021,2020-08-27 14:22:17.833,14,2020,2020Q3,2020-08,2020-08-01,Thursday,4,0,2020-08-31 18:22:17.833,2020-08-01
3,VCV00014741/082021,2020-08-28 00:32:20.523,0,2020,2020Q3,2020-08,2020-08-01,Friday,5,0,2020-09-01 04:32:20.523,2020-09-01
4,AEIBK2027446,2020-08-26 09:55:08.000,9,2020,2020Q3,2020-08,2020-08-01,Wednesday,3,0,2020-08-30 13:55:08.000,2020-08-01
5,AEIBK2027346,2020-08-25 15:45:33.000,15,2020,2020Q3,2020-08,2020-08-01,Tuesday,2,0,2020-08-29 19:45:33.000,2020-08-01
6,VCV00014935/082021,2020-08-28 11:14:21.960,11,2020,2020Q3,2020-08,2020-08-01,Friday,5,0,2020-09-01 15:14:21.960,2020-09-01
7,VCV00014113/082021,2020-08-27 15:05:12.770,15,2020,2020Q3,2020-08,2020-08-01,Thursday,4,0,2020-08-31 19:05:12.770,2020-08-01
8,VCV00013971/082021,2020-08-27 12:57:32.070,12,2020,2020Q3,2020-08,2020-08-01,Thursday,4,0,2020-08-31 16:57:32.070,2020-08-01
9,AEIBK2027316,2020-08-25 13:39:48.000,13,2020,2020Q3,2020-08,2020-08-01,Tuesday,2,0,2020-08-29 17:39:48.000,2020-08-01


#### __1.3 Route và Distnace Features__
Phần này tạo nhằm hỗ trợ phân tích delivery performance

##### Tạo Route
- Route có grain:
Origin Location $\rightarrow$ Destination Location

In [20]:
df_analytical['route'] = (
    df_analytical['origin_location'].astype('string')
    + ' -> '
    + df_analytical['destination_location'].astype('string')
)

df_analytical[
    [
        'booking_id',
        'origin_location',
        'destination_location',
        'route'
    ]
].head(5)

,booking_id,origin_location,destination_location,route
0,AEIBK2027469,"Shive, pune, maharashtra","Pondur, Kanchipuram, Tamil Nadu","Shive, pune, maharashtra -> Pondur, Kanchipura..."
1,VCV00014153/082021,"Daimler India Commercial Vehicles,Kanchipuram,...","Daimler India Commercial Vehicles,Kanchipuram,...","Daimler India Commercial Vehicles,Kanchipuram,..."
2,VCV00014063/082021,"Daimler India Commercial Vehicles,Kanchipuram,...","Daimler India Commercial Vehicles,Kanchipuram,...","Daimler India Commercial Vehicles,Kanchipuram,..."
3,VCV00014741/082021,"Daimler India Commercial Vehicles,Kanchipuram,...","Daimler India Commercial Vehicles,Kanchipuram,...","Daimler India Commercial Vehicles,Kanchipuram,..."
4,AEIBK2027446,"Khorajnanoda, ahmedabad, gujarat","Singaperumalkoil, Kanchipuram, Tamil Nadu","Khorajnanoda, ahmedabad, gujarat -> Singaperum..."


In [21]:
# Kiểm tra route volume
route_volume = (
    df_analytical["route"].value_counts().rename("booking_count").to_frame())

route_volume.head(5)

,booking_count
route,
"Daimler India Commercial Vehicles,Kanchipuram,Tamil Nadu -> Daimler India Commercial Vehicles,Kanchipuram,Tamil Nadu",308
"Shive, pune, maharashtra -> Pondur, Kanchipuram, Tamil Nadu",127
"Jamalpur, gurgaon, haryana -> Singaperumalkoil, Kanchipuram, Tamil Nadu",126
"Khorajnanoda, ahmedabad, gujarat -> Singaperumalkoil, Kanchipuram, Tamil Nadu",103
"Ashok Leyland Plant 2-Hosur,Hosur,Karnataka -> Ashok Leyland Plant 2-Hosur,Hosur,Karnataka",94


##### Kiểm tra lại distance distribution ở booking-level

In [22]:
distance = (df_analytical["transportation_distance_km"])
distance.describe()

count    3434.000000
mean      840.855213
std       852.074642
min         0.000000
25%       106.250000
50%       400.000000
75%      1290.000000
max      2898.000000
Name: transportation_distance_km, dtype: float64

In [23]:
distance_quantiles = (distance.dropna().quantile([   
            0.25,
            0.50,
            0.75]))

distance_quantiles

0.25     106.25
0.50     400.00
0.75    1290.00
Name: transportation_distance_km, dtype: float64

In [24]:
# Lưu lại các ngưỡng
distance_q1 = distance_quantiles.loc[0.25]
distance_median = distance_quantiles.loc[0.50]
distance_q3 = distance_quantiles.loc[0.75]

print(f"Q1: {distance_q1:.2f} km")
print(f"Median: {distance_median:.2f} km")
print(f"Q3: {distance_q3:.2f} km")

Q1: 106.25 km
Median: 400.00 km
Q3: 1290.00 km


In [25]:
# Dùng các quartile của booking-level distance để tạo nhóm có quy mô tương đối cân bằng.
df_analytical['distance_band'] = pd.cut(
    df_analytical['transportation_distance_km'],
    bins=[
        float('-inf'),
        distance_q1,
        distance_median,
        distance_q3,
        float('inf')
    ],
    labels=[
        'Short',
        'Medium',
        'Long',
        'Very Long'
    ],
    include_lowest=True
).astype('string').fillna('Unknown')

In [26]:
df_analytical[
    "distance_band"
].value_counts(
    dropna=False
).sort_index()

distance_band
Long         856
Medium       862
Short        859
Unknown      148
Very Long    857
Name: count, dtype: Int64

In [27]:
DISTANCE_BAND_ORDER = {
    'Unknown': 0,
    'Short': 1,
    'Medium': 2,
    'Long': 3,
    'Very Long': 4
}

df_analytical['distance_band_order'] = (
    df_analytical['distance_band']
    .map(DISTANCE_BAND_ORDER)
    .astype('Int64')
)

assert df_analytical['distance_band_order'].notna().all()

In [28]:
# So sánh location bằng normalized key thay vì so sánh text nguyên bản.
# Hai cột origin/destination có thể khác cách viết hoa dù cùng một địa điểm.
def normalize_location_key(series):
    return (
        series.astype('string')
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
        .str.casefold()
    )

origin_location_key = normalize_location_key(
    df_analytical['origin_location']
)
destination_location_key = normalize_location_key(
    df_analytical['destination_location']
)

df_analytical['same_origin_destination'] = (
    origin_location_key.eq(destination_location_key).astype('Int64')
)

df_analytical['is_zero_distance'] = (
    df_analytical['transportation_distance_km'].eq(0).astype('Int64')
)

route_distance_check = pd.crosstab(
    df_analytical['same_origin_destination'],
    df_analytical['is_zero_distance'],
    rownames=['same_origin_destination'],
    colnames=['is_zero_distance']
)

route_distance_check

is_zero_distance,0,1
same_origin_destination,,
0,2992,0
1,572,18


In [29]:
df_analytical[
    [
        'booking_id',
        'origin_location',
        'destination_location',
        'route',
        'transportation_distance_km',
        'distance_band',
        'distance_band_order',
        'same_origin_destination',
        'is_zero_distance'
    ]
].sample(5, random_state=42)

,booking_id,origin_location,destination_location,route,transportation_distance_km,distance_band,distance_band_order,same_origin_destination,is_zero_distance
1407,AEIBK2025084,"Jamalpur, gurgaon, haryana","Singaperumalkoil, Kanchipuram, Tamil Nadu","Jamalpur, gurgaon, haryana -> Singaperumalkoil...",2425.0,Very Long,4,0,0
411,AEIBK2026867,"Khorajnanoda, ahmedabad, gujarat","Jamalpur, Gurgaon, Haryana","Khorajnanoda, ahmedabad, gujarat -> Jamalpur, ...",900.0,Long,3,0,0
802,VCV00008782/082021,"Ashok Leyland Plant 2-Hosur,Hosur,Karnataka","Ashok Leyland Plant 2-Hosur,Hosur,Karnataka","Ashok Leyland Plant 2-Hosur,Hosur,Karnataka ->...",110.0,Medium,2,1,0
3032,AEIBK2012507,"Sonai, kolkata, west bengal","Kataganj, Nadia, West Bengal","Sonai, kolkata, west bengal -> Kataganj, Nadia...",51.0,Short,1,0,0
1546,AEIBK2020731,"Embalam, pondicherry, pondicherry","Jamalpur, Gurgaon, Haryana","Embalam, pondicherry, pondicherry -> Jamalpur,...",2700.0,Very Long,4,0,0


#### __1.4 Transit Time và Distance Integrity__
Phần này tạo các feature kiểm chứng tính nhất quán giữa `transportation_distance_km` và thời gian di chuyển thực tế.

Lý do: crosstab ở mục 1.3 cho thấy 572 booking có `same_origin_destination = 1` nhưng distance vẫn lớn hơn 0. Cờ `is_zero_distance` chỉ bắt được 18 trong số 590 booking cùng điểm đi - đến, nên cần một tiêu chí kiểm tra mạnh hơn.


In [30]:
# actual_transit_hours la do dai chuyen di thuc te, khac delivery_variance_hours
# (bien do so Actual ETA voi Planned ETA, khong phai do dai chuyen di).
df_analytical['actual_transit_hours'] = (
    (df_analytical['actual_eta'] - df_analytical['trip_start_date'])
    .dt.total_seconds()
    .div(3600)
    .astype('Float64')
)

# Chi chia khi thoi luong duong de tranh inf va chia cho 0.
positive_transit = df_analytical['actual_transit_hours'].where(
    df_analytical['actual_transit_hours'] > 0
)

df_analytical['implied_speed_kmph'] = (
    df_analytical['transportation_distance_km']
    .div(positive_transit)
    .astype('Float64')
)

print("Booking co thoi luong chuyen di <= 0 gio:",
      (df_analytical['actual_transit_hours'] <= 0).sum())

df_analytical['implied_speed_kmph'].describe()


Booking co thoi luong chuyen di <= 0 gio: 7


count        3405.0
mean      44.441918
std      209.978015
min             0.0
25%        1.449175
50%        7.103527
75%       17.797474
max      4738.71908
Name: implied_speed_kmph, dtype: Float64

##### Tạo cờ `distance_time_inconsistent`
Vận tốc suy ra (`implied_speed_kmph`) chỉ dùng để kiểm tra tính hợp lý, không dùng làm KPI.

Chỉ đặt ngưỡng ở cận trên: xe vận tải không thể duy trì trung bình trên 90 km/h cho toàn chuyến, nên vượt mức này nghĩa là distance không khớp với thời gian thực tế. Không đặt ngưỡng cận dưới vì `actual_transit_hours` bao gồm cả thời gian chờ và dwell, nên vận tốc thấp là bình thường.


In [31]:
MAX_PLAUSIBLE_SPEED_KMPH = 90

df_analytical['distance_time_inconsistent'] = (
    df_analytical['implied_speed_kmph']
    .gt(MAX_PLAUSIBLE_SPEED_KMPH)
    .fillna(False)
    .astype('Int64')
)

inconsistency_check = pd.crosstab(
    df_analytical['same_origin_destination'],
    df_analytical['distance_time_inconsistent'],
    rownames=['same_origin_destination'],
    colnames=['distance_time_inconsistent']
)

inconsistency_check


distance_time_inconsistent,0,1
same_origin_destination,,
0,2894,98
1,432,158


##### Tạo `route_type`
Cờ nhị phân `same_origin_destination` gộp nhiều trường hợp có bản chất khác nhau vào cùng một nhóm, nên thay bằng phân loại chi tiết hơn:

- `Line Haul`: tuyến vận chuyển bình thường, distance dùng được cho phân tích.
- `Intra Facility`: cùng nhãn địa điểm và thời lượng quá ngắn so với distance. Đây là di chuyển nội bộ hoặc shunting trong bãi, con số km là giá trị master data của lane chứ không phải quãng đường của chuyến đó.
- `Same Label`: cùng nhãn địa điểm nhưng thời lượng hợp lý, có thể là milk-run vòng tròn thật.
- `Suspect Distance`: origin khác destination nhưng distance vẫn không khớp thời gian.
- `Zero Distance`: cùng địa điểm và distance bằng 0.
- `Unknown`: thiếu distance.


In [32]:
same_od = df_analytical['same_origin_destination'].eq(1)
zero_distance = df_analytical['is_zero_distance'].eq(1)
missing_distance = df_analytical['transportation_distance_km'].isna()
inconsistent = df_analytical['distance_time_inconsistent'].eq(1)
very_short_trip = df_analytical['actual_transit_hours'].lt(1).fillna(False)

route_type_conditions = [
    missing_distance,
    same_od & zero_distance,
    same_od & (inconsistent | very_short_trip),
    same_od,
    inconsistent
]

route_type_choices = [
    'Unknown',
    'Zero Distance',
    'Intra Facility',
    'Same Label',
    'Suspect Distance'
]

df_analytical['route_type'] = pd.Series(
    np.select(route_type_conditions, route_type_choices, default='Line Haul'),
    index=df_analytical.index,
    dtype='string'
)

df_analytical['route_type'].value_counts(dropna=False)


route_type
Line Haul           2754
Same Label           396
Intra Facility       168
Unknown              148
Suspect Distance      98
Zero Distance         18
Name: count, dtype: Int64

##### Tạo cờ `analysis_distance_valid`
Cờ duy nhất mà mọi phân tích theo km cần lọc theo, để không phải viết lại điều kiện ở từng chỗ.

`distance_band` cũng được đặt lại thành `Unknown` cho các booking không đạt cờ này, vì nếu giữ nguyên thì các chuyến di chuyển nội bộ vài chục phút sẽ bị xếp cùng nhóm với line-haul thật và làm nhiễu phân tích on-time theo distance band.


In [33]:
df_analytical['analysis_distance_valid'] = (
    df_analytical['route_type'].eq('Line Haul').astype('Int64')
)

# distance_band chỉ có ý nghĩa trên các tuyến có distance đáng tin
df_analytical.loc[
    df_analytical['analysis_distance_valid'].eq(0),
    ['distance_band', 'distance_band_order']
] = ['Unknown', 0]

print(
    "Booking dung duoc cho KPI theo km:",
    int(df_analytical['analysis_distance_valid'].sum()),
    "/",
    len(df_analytical)
)

# On-time vẫn được tính cho nhóm bị loại, miễn là time line hợp lệ
print(
    "Booking bi loai khoi KPI km nhung van hop le cho on-time:",
    int((df_analytical['analysis_distance_valid'].eq(0) & eligible_mask).sum())
)

assert df_analytical['distance_band_order'].notna().all()

df_analytical['distance_band'].value_counts(dropna=False).sort_index()


Booking dung duoc cho KPI theo km: 2754 / 3582
Booking bi loai khoi KPI km nhung van hop le cho on-time: 823


distance_band
Long         776
Medium       581
Short        563
Unknown      828
Very Long    834
Name: count, dtype: Int64

##### Kiểm chứng lại các booking mâu thuẫn

In [34]:
# So sanh profile giua cac route_type de xac nhan phan loai hop ly.
df_analytical.groupby('route_type')[
    [
        'transportation_distance_km',
        'actual_transit_hours',
        'implied_speed_kmph'
    ]
].median().round(2)


,transportation_distance_km,actual_transit_hours,implied_speed_kmph
route_type,,,
Intra Facility,110.0,0.45,239.61
Line Haul,900.0,130.85,6.49
Same Label,69.0,17.29,3.97
Suspect Distance,535.0,1.8,191.72
Unknown,NaN,240.87,<NA>
Zero Distance,0.0,200.66,0.0


In [35]:
# TH đại diện: origin = destination = Ashok Leyland Plant 2-Hosur,
# distance 110 km nhưng chuyến đi chỉ kéo dài khoảng 20 phút
df_analytical[
    df_analytical['booking_id'].eq('VCV00008782/082021')
][
    [
        'booking_id',
        'origin_location',
        'destination_location',
        'transportation_distance_km',
        'actual_transit_hours',
        'implied_speed_kmph',
        'route_type',
        'distance_band',
        'analysis_distance_valid'
    ]
]


,booking_id,origin_location,destination_location,transportation_distance_km,actual_transit_hours,implied_speed_kmph,route_type,distance_band,analysis_distance_valid
802,VCV00008782/082021,"Ashok Leyland Plant 2-Hosur,Hosur,Karnataka","Ashok Leyland Plant 2-Hosur,Hosur,Karnataka",110.0,0.343545,320.190691,Intra Facility,Unknown,0


In [36]:
df_analytical[
    [
        'booking_id',
        'route',
        'transportation_distance_km',
        'actual_transit_hours',
        'implied_speed_kmph',
        'distance_time_inconsistent',
        'route_type',
        'distance_band',
        'analysis_distance_valid'
    ]
].sample(5, random_state=42)


,booking_id,route,transportation_distance_km,actual_transit_hours,implied_speed_kmph,distance_time_inconsistent,route_type,distance_band,analysis_distance_valid
1407,AEIBK2025084,"Jamalpur, gurgaon, haryana -> Singaperumalkoil...",2425.0,52.753356,45.96864,0,Line Haul,Very Long,1
411,AEIBK2026867,"Khorajnanoda, ahmedabad, gujarat -> Jamalpur, ...",900.0,2.062289,436.408306,1,Suspect Distance,Unknown,0
802,VCV00008782/082021,"Ashok Leyland Plant 2-Hosur,Hosur,Karnataka ->...",110.0,0.343545,320.190691,1,Intra Facility,Unknown,0
3032,AEIBK2012507,"Sonai, kolkata, west bengal -> Kataganj, Nadia...",51.0,1366.905117,0.037311,0,Line Haul,Short,1
1546,AEIBK2020731,"Embalam, pondicherry, pondicherry -> Jamalpur,...",2700.0,1183.529148,2.281313,0,Line Haul,Very Long,1


__Kết luận:__
- `transportation_distance_km` là giá trị master data gán theo cặp origin - destination, không phải quãng đường đo được của từng chuyến. Bằng chứng: 465 trên 521 cặp origin - destination chỉ có duy nhất một giá trị distance, và toàn bộ 94 booking cùng điểm đi - đến tại Ashok Leyland Plant 2-Hosur đều mang đúng 110.0 km trong khi thời lượng chuyến trung vị chỉ 0.36 giờ.
- `distance_time_inconsistent` phát hiện 256 booking mâu thuẫn, so với 18 booking mà `is_zero_distance` bắt được.
- Mọi KPI theo km cần lọc `analysis_distance_valid = 1`. Các booking còn lại vẫn dùng được cho KPI on-time nếu `valid_kpi_timeline` bằng True, vì `planned_eta` và `actual_eta` của chúng không bị ảnh hưởng.


#### __1.5 Delay Severity__

Ở phần này, ta sẽ phân loại mức độ giao trễ để giúp đánh giá không chỉ booking có bị delay hay không mà còn xác định mức độ nghiêm trọng của delay.

Quy ước:
- `On Time`: không trễ.
- `Slight Delay`: trễ trên 0 đến 24 giờ.
- `Medium Delay`: trễ trên 24 đến 72 giờ.
- `Severe Delay`: trễ trên 72 giờ.
- `Unknown`: booking không đủ điều kiện tính KPI.

In [37]:
conditions = [
    ~eligible_mask,
    eligible_mask & df_analytical["delay_hours"].eq(0),
    eligible_mask & df_analytical["delay_hours"].gt(0) & df_analytical["delay_hours"].le(24),
    eligible_mask & df_analytical["delay_hours"].gt(24) & df_analytical["delay_hours"].le(72),
    eligible_mask & df_analytical["delay_hours"].gt(72)
]

choices = [
    "Unknown",
    "On Time",
    "Slight Delay",
    "Medium Delay",
    "Severe Delay"
]

df_analytical["delay_level"] = pd.Series(
    np.select(conditions, choices, default="Unknown"),
    index=df_analytical.index,
    dtype="string"
)

df_analytical["delay_level"].value_counts(dropna=False)

delay_level
On Time         1364
Severe Delay    1362
Medium Delay     532
Slight Delay     290
Unknown           34
Name: count, dtype: Int64

##### Severe Delay 


In [38]:
df_analytical["is_severe_delay"] = pd.Series(
    pd.NA, index=df_analytical.index, dtype="Int64")

df_analytical.loc[eligible_mask, "is_severe_delay"] = (
    df_analytical.loc[eligible_mask, "delay_hours"].gt(72).astype("Int64"))

df_analytical["is_severe_delay"].value_counts(dropna=False)

is_severe_delay
0       2186
1       1362
<NA>      34
Name: count, dtype: Int64

#### __1.6 Tính các KPIs baseline__

In [39]:
# Tổng số booking
total_bookings = len(df_analytical)

# Booking đủ điều kiện tính KPI
valid_kpi_bookings = eligible_mask.sum()

# Booking giao trễ
delayed_bookings = df_analytical['is_delayed'].eq(1).sum()

# Booking đúng hạn
ontime_bookings = df_analytical['is_delayed'].eq(0).sum()

# Delay Rate
delay_rate = delayed_bookings / valid_kpi_bookings

# On-Time Rate
ontime_rate = ontime_bookings / valid_kpi_bookings

# Severe Delay Rate
severe_delay_bookings = df_analytical["is_severe_delay"].eq(1).sum()
severe_delay_rate = severe_delay_bookings / valid_kpi_bookings

kpi_baseline = pd.Series({
    "Total Bookings": total_bookings,
    "Valid KPI Bookings": valid_kpi_bookings,
    "Delayed Bookings": delayed_bookings,
    "On-Time Bookings": ontime_bookings,
    "Severe Delay Bookings": severe_delay_bookings,
    "Delay Rate": round(delay_rate * 100, 2),
    "On-Time Rate": round(ontime_rate * 100, 2),
    "Severe Delay Rate": round(severe_delay_rate * 100, 2)
})

kpi_baseline

Total Bookings           3582.00
Valid KPI Bookings       3548.00
Delayed Bookings         2184.00
On-Time Bookings         1364.00
Severe Delay Bookings    1362.00
Delay Rate                 61.56
On-Time Rate               38.44
Severe Delay Rate          38.39
dtype: float64

> Ta thấy rằng tỉ lệ đúng giờ chỉ chiếm **38.44**% trong khi tỉ lệ delay rất cao **61.56**%

##### Thống kê mức độ nghiêm trọng của delay

In [40]:
positive_delay = df_analytical.loc[
    df_analytical["is_delayed"].eq(1),
    "delay_hours"
]

mean_delay = positive_delay.mean()
median_delay = positive_delay.median()
p90_delay = positive_delay.quantile(0.90)
p95_delay = positive_delay.quantile(0.95)
total_delay_hours = positive_delay.sum()

delay_severity_metrics = pd.Series({
    "Mean Positive Delay (h)": round(mean_delay, 2),
    "Median Positive Delay (h)": round(median_delay, 2),
    "P90 Delay (h)": round(p90_delay, 2),
    "P95 Delay (h)": round(p95_delay, 2),
    "Total Delay Hours": round(total_delay_hours, 2)
})

delay_severity_metrics

Mean Positive Delay (h)         275.46
Median Positive Delay (h)       116.44
P90 Delay (h)                   616.77
P95 Delay (h)                  1099.29
Total Delay Hours            601608.45
dtype: float64

>Kết quả cho thấy tình trạng giao trễ không chỉ phổ biến mà còn có mức độ nghiêm trọng cao.

>- Median Positive Delay là **116,44 giờ**, cho thấy một booking bị trễ điển hình chậm khoảng 4,9 ngày.
>- Mean Positive Delay là **275,46 giờ**, cao hơn đáng kể so với median, cho thấy phân phối delay bị lệch phải bởi các trường hợp trễ rất lớn.
>- P90 Delay đạt **616,77 giờ**, nghĩa là 10% booking bị trễ có thời gian delay lớn hơn khoảng 25,7 ngày.
>- P95 Delay đạt **1.099,29 giờ**, cho thấy 5% booking bị trễ nghiêm trọng nhất vượt khoảng 45,8 ngày.
>- Tổng thời gian delay của các booking bị trễ là **601.608,45 giờ**.

>Do phân phối delay có đuôi dài, Median, P90 và P95 nên được sử dụng cùng với Mean để đánh giá hiệu suất giao hàng thay vì chỉ dựa vào giá trị trung bình.

In [41]:
unknown_kpi_bookings = total_bookings - valid_kpi_bookings

kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Bookings",
        "Valid KPI Bookings",
        "Delayed Bookings",
        "On-Time Bookings",
        "Unknown KPI Bookings",
        "Delay Rate",
        "On-Time Rate",
        "Mean Positive Delay (h)",
        "Median Positive Delay (h)",
        "P90 Delay (h)",
        "P95 Delay (h)",
        "Severe Delay Bookings",
        "Severe Delay Rate",
        "Total Delay Hours"
    ],
    
    "Value": [
        total_bookings,
        valid_kpi_bookings,
        delayed_bookings,
        ontime_bookings,
        unknown_kpi_bookings,
        round(delay_rate * 100, 2),
        round(ontime_rate * 100, 2),
        round(mean_delay, 2),
        round(median_delay, 2),
        round(p90_delay, 2),
        round(p95_delay, 2),
        severe_delay_bookings,
        round(severe_delay_rate * 100, 2),
        round(total_delay_hours, 2)
    ]
})

kpi_summary

,KPI,Value
0,Total Bookings,3582.00
1,Valid KPI Bookings,3548.00
2,Delayed Bookings,2184.00
3,On-Time Bookings,1364.00
4,Unknown KPI Bookings,34.00
5,Delay Rate,61.56
6,On-Time Rate,38.44
7,Mean Positive Delay (h),275.46
8,Median Positive Delay (h),116.44
9,P90 Delay (h),616.77


In [44]:
from pathlib import Path

OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

feature_data_path = OUTPUT_DIR / "logistics_tracking_featured.pkl"
kpi_summary_path = OUTPUT_DIR / "kpi_summary.csv"

# Lưu dataset sau Feature Engineering
df_analytical.to_pickle(feature_data_path)

# Lưu bảng KPI
kpi_summary.to_csv(
    kpi_summary_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved feature dataset:", feature_data_path)
print("Saved KPI summary:", kpi_summary_path)
print("Dataset shape:", df_analytical.shape)

Saved feature dataset: ..\data\processed\logistics_tracking_featured.pkl
Saved KPI summary: ..\data\processed\kpi_summary.csv
Dataset shape: (3582, 49)
